## main libraries

In [ ]:
import torch
from torch import nn

# vision
import torchvision
from torchvision import datasets
from torchvision.transforms import ToTensor

import matplotlib.pyplot as plt


# dataset

In [ ]:
training_data=datasets.FashionMNIST(
    root="data", # location 
    train=True, # bool -> for training or test
    download=True,
    transform=ToTensor(), # turning or transforing images PLT -> tensor
    target_transform=None # even output can convet to a tensor
)


testing_data=datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor()
        

)

In [ ]:
image , lable=training_data[100]

# image ---> the actual metrice of the image with PIL tensor format
# lable ---> it is an int that shows which object it is range:[0, len of classs-1]
image.shape , lable
# if [3,h,w] then ---> a 3d cube that z is color of  and rgb but 
#1 means a black and white picture


In [ ]:
len(training_data)

In [ ]:
Label_names=training_data.classes
Label_names.__len__(),Label_names


# visual

In [ ]:
plt.imshow(image.squeeze(),cmap="plasma")
plt.title(Label_names[lable])

## dataloader

In [ ]:
from torch.utils.data import DataLoader


BATCH=32 

# spliting datas to ---> small batches
training_dataloader=DataLoader(training_data,
                               batch_size=BATCH,
                               shuffle=True)


testing_dataloader=DataLoader(testing_data,
                               batch_size=BATCH,
                               shuffle=False)

f"{training_dataloader.__len__()} of {BATCH} batches"
    


# model 0

In [ ]:
class Fashion(nn.Module):
    def __init__(self, input_shape: int, hidden_units: int, output_shape: int):
        super().__init__()
        self.layer_stack = nn.Sequential(
            nn.Flatten(), # converting [height , width] --> [ height * width]
            nn.Linear(in_features=input_shape, out_features=hidden_units), # in_features = number of features in a data sample (784 pixels)
            nn.Linear(in_features=hidden_units, out_features=output_shape)
        )
    
    def forward(self, x):
        return self.layer_stack(x) 
# input is 784 ---> 28 * 28
#output is len(lable_class)

model_0=Fashion(input_shape=784,
               hidden_units=10,
               output_shape=10)
model_0.to("cpu")
        

In [ ]:
import requests
from pathlib import Path 

# Download helper functions from Learn PyTorch repo (if not already downloaded)
if Path("helper_functions.py").is_file():
  print("helper_functions.py already exists, skipping download")
else:
  print("Downloading helper_functions.py")
  # Note: you need the "raw" GitHub URL for this to work
  request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
  with open("helper_functions.py", "wb") as f:
    f.write(request.content)

In [ ]:
from helper_functions import accuracy_fn

loss_func=nn.CrossEntropyLoss()# need notes for this loss func
optim=torch.optim.SGD(params=model_0.parameters(),
                      lr=0.1)


In [ ]:
from tqdm.auto import tqdm
epochs= 4


for epoch in tqdm(range(epochs)):
    train_loss=0
    # trining through batches
    for batch,(X,y) in enumerate(training_dataloader):
        model_0.train()

        y_predict=model_0(X)
        loss=loss_func(y_predict,y)

        train_loss+=loss

        optim.zero_grad()

        loss.backward()

        optim.step()

        if batch % 1000 == 0:
            print(f"Looked at {batch * len(X)}/{len(training_dataloader)} samples")
    train_loss /= len(training_dataloader)
    test_loss,test_acc=0, 0
    
    model_0.eval()
    with torch.inference_mode():
        for X,y in testing_dataloader:
            test_predict=model_0(X)

            test_loss +=loss_func(test_predict,y)
            test_acc +=accuracy_fn(y_true=y, y_pred=test_predict.argmax(dim=1))

            test_loss/= len(testing_dataloader)
            test_acc/= len(testing_dataloader)
            print(f"\nTrain loss: {train_loss:.5f} | Test loss: {test_loss:.5f}, Test acc: {test_acc:.2f}%\n")
    